# MIMIC-IV Multimodal Data Preprocessing

This notebook contains the preprocessing pipeline for matching and organizing MIMIC-IV multimodal data (EHR, CXR, ECG, Notes) into sequential patient records.


## Goals and Matching Strategy

### Goal
Create a unified patient record structure that organizes all multimodal data (EHR, CXR, ECG, Notes) by admission (hadm_id) for sequential processing.

### Matching Strategy

**Primary Structure:**
```
Patient (subject_id)
  └── Admission (hadm_id)
      ├── EHR Data (diagnoses, procedures, labs, prescriptions)
      ├── Discharge Notes
      ├── CXR Studies (matched by time)
      ├── Radiology Notes (matched by time + content)
      └── ECG Records (matched by time)
```

**Key Identifiers:**
- `subject_id`: Common identifier across ALL modalities (links everything)
- `hadm_id`: Hospital admission ID (primary grouping unit for EHR data)
- `study_id`: Modality-specific (CXR study_id ≠ ECG study_id)

**Matching Rules:**

1. **EHR Data → Admission**: Direct link via `hadm_id` (all EHR tables have hadm_id)
2. **Discharge Notes → Admission**: Direct link via `hadm_id` (1:1 relationship)
3. **CXR Studies → Admission**: Temporal matching
   - Match if: `subject_id` matches AND `admittime <= CXR_time <= dischtime`
   - CXR_time = `StudyDate + StudyTime`
4. **Radiology Notes → CXR → Admission**: 
   - Filter for chest-related notes (text contains CHEST/CXR)
   - Match by `subject_id` + temporal proximity (within 24 hours)
   - Then link to admission via time window
5. **ECG Records → Admission**: Temporal matching
   - Match if: `subject_id` matches AND `admittime <= ecg_time <= dischtime`

**Output Format:**
- Each record represents one admission with all matched modalities
- Stored with flags to identify which modalities are present
- Enables sequential processing without re-running matching


In [21]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
import json
import pickle
import zipfile
from pathlib import Path
from tqdm import tqdm
from typing import Optional, Dict, List, Tuple
from datetime import timedelta
from collections import defaultdict

# Set base paths - all data from xiaochen
BASE_DATA_PATH = Path("/data/xiaochen/physionet.org/files/")
MIMICIV_PATH = BASE_DATA_PATH / "mimiciv"
MIMIC_CXR_JPG_PATH = BASE_DATA_PATH / "mimic-cxr-jpg"
MIMIC_CXR_PATH = BASE_DATA_PATH / "mimic-cxr"  # Original CXR with .txt files
MIMIC_CXR_REPORTS_ZIP_PATH = MIMIC_CXR_PATH / "2.0.0" / "mimic-cxr-reports.zip"
MIMIC_IV_ECG_PATH = BASE_DATA_PATH / "mimic-iv-ecg"
MIMIC_IV_NOTE_PATH = BASE_DATA_PATH / "mimic-iv-note"

OUTPUT_PATH = Path("./output/")
OUTPUT_PATH.mkdir(exist_ok=True)
V0_PATH = OUTPUT_PATH / "v0"
V1_PATH = OUTPUT_PATH / "v1"
V0_PATH.mkdir(exist_ok=True)
V1_PATH.mkdir(exist_ok=True)

# Matching results cache path
MATCHING_CACHE_PATH = OUTPUT_PATH / "matching_results.pkl"
MATCHING_STATS_PATH = OUTPUT_PATH / "matching_stats.json"
PATIENT_UNASSIGNED_PATH = OUTPUT_PATH / "patient_unassigned.pkl"
ECG_OFFSET_REPORT_PATH = OUTPUT_PATH / "ecg_offset_report.json"
CXR_REPORT_CACHE_PATH = OUTPUT_PATH / "cxr_reports.pkl"

# Versioned paths
V0_MATCHING_PATH = V0_PATH / "matching_results.pkl"
V0_STATS_PATH = V0_PATH / "matching_stats.json"
V0_UNASSIGNED_PATH = V0_PATH / "patient_unassigned.pkl"
V0_ECG_OFFSET_PATH = V0_PATH / "ecg_offset_report.json"

V1_MATCHING_PATH = V1_PATH / "matching_results.pkl"
V1_STATS_PATH = V1_PATH / "matching_stats.json"

print("MIMIC-IV Preprocessing Pipeline")
print(f"Base data path: {BASE_DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")


MIMIC-IV Preprocessing Pipeline
Base data path: /data/xiaochen/physionet.org/files
Output path: output


## Data Loading Functions

Functions to load data from different MIMIC-IV subsets.


In [22]:
def load_ehr_data(lazy: bool = False):
    """
    Load MIMIC-IV EHR data.
    
    Args:
        lazy: If True, only load essential files (admissions, patients, diagnoses)
        
    Returns:
        Tuple of dataframes: (admissions, patients, diagnoses, prescriptions, labevents, procedures)
    """
    print("Loading MIMIC-IV EHR dataset files...")
    path = MIMICIV_PATH / "2.2" / "hosp"
    
    print("  Loading admissions...")
    admissions = pd.read_csv(path / "admissions.csv")
    print("  Loading patients...")
    patients = pd.read_csv(path / "patients.csv")
    print("  Loading diagnoses...")
    diagnoses = pd.read_csv(path / "diagnoses_icd.csv")
    
    # Convert column names to lowercase for consistency
    admissions.columns = admissions.columns.str.lower()
    patients.columns = patients.columns.str.lower()
    diagnoses.columns = diagnoses.columns.str.lower()
    
    # Convert datetime columns
    admissions['admittime'] = pd.to_datetime(admissions['admittime'])
    admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
    
    if lazy:
        return admissions, patients, diagnoses, None, None, None
    
    print("  Loading prescriptions...")
    prescriptions = pd.read_csv(path / "prescriptions.csv")
    prescriptions.columns = prescriptions.columns.str.lower()
    prescriptions_cleaned = prescriptions.dropna(subset=['formulary_drug_cd', 'subject_id'])
    print(f"  Dropped {prescriptions.shape[0] - prescriptions_cleaned.shape[0]} rows with missing data.")
    
    print("  Loading labevents...")
    labevents = pd.read_csv(path / "labevents.csv")
    labevents.columns = labevents.columns.str.lower()
    labevents['itemid'] = labevents['itemid'].astype(str)
    
    print("  Loading procedures...")
    procedures = pd.read_csv(path / "procedures_icd.csv")
    procedures.columns = procedures.columns.str.lower()
    
    print("✅ EHR data loaded successfully.")
    return admissions, patients, diagnoses, prescriptions_cleaned, labevents, procedures


In [23]:
def load_cxr_data(lazy: bool = False):
    """
    Load MIMIC-CXR dataset files.
    
    Args:
        lazy: If True, only load metadata
        
    Returns:
        Tuple of dataframes: (metadata, chexpert, negbio, split)
    """
    print("Loading MIMIC-CXR dataset files...")
    
    # Version 2.0.0
    jpg_root = MIMIC_CXR_JPG_PATH / "2.0.0" / "files"
    metadata_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-metadata.csv"
    chexpert_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-chexpert.csv"
    negbio_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-negbio.csv"
    split_path = MIMIC_CXR_JPG_PATH / "2.0.0" / "mimic-cxr-2.0.0-split.csv"
    
    print("  Loading metadata...")
    metadata = pd.read_csv(metadata_path)
    
    # Convert StudyDate and StudyTime to datetime
    metadata['StudyDate_dt'] = pd.to_datetime(
        metadata['StudyDate'].astype(str), 
        format='%Y%m%d', 
        errors='coerce'
    )
    # Parse StudyTime (HHMMSS.SSS format)
    metadata['StudyTime_parsed'] = pd.to_timedelta(
        metadata['StudyTime'].fillna(0).astype(int).astype(str).str.zfill(6).str[:2] + ':' + 
        metadata['StudyTime'].fillna(0).astype(int).astype(str).str.zfill(6).str[2:4] + ':' + 
        metadata['StudyTime'].fillna(0).astype(int).astype(str).str.zfill(6).str[4:6], 
        errors='coerce'
    )
    metadata['study_datetime'] = metadata['StudyDate_dt'] + metadata['StudyTime_parsed']
    
    if lazy:
        return metadata, None, None, None
    
    print("  Loading chexpert...")
    chexpert = pd.read_csv(chexpert_path)
    
    print("  Loading negbio...")
    negbio = pd.read_csv(negbio_path)
    
    print("  Loading split...")
    split = pd.read_csv(split_path)
    
    print("✅ CXR data loaded successfully.")
    return metadata, chexpert, negbio, split


In [24]:
def load_cxr_reports(cache_path: Path = CXR_REPORT_CACHE_PATH, force_reload: bool = False) -> Dict[int, str]:
    """Load MIMIC-CXR study reports (final reports) as a mapping of study_id -> text.
    
    Reads individual .txt files from the mimic-cxr directory structure.
    Example: /data/xiaochen/physionet.org/files/mimic-cxr/2.0.0/files/p19/p19000065/s51613820.txt

    Args:
        cache_path: Optional path to cache the study_id -> report mapping.
        force_reload: If True, ignore any existing cache and rebuild from source.

    Returns:
        Dictionary mapping study_id (int) to the corresponding radiology report text.
    """
    if not force_reload and cache_path is not None and cache_path.exists():
        print(f"Loading cached CXR reports from {cache_path} ...")
        with cache_path.open('rb') as f:
            reports = pickle.load(f)
        print(f"  Loaded {len(reports):,} cached CXR reports.")
        return reports

    cxr_files_path = MIMIC_CXR_PATH / "2.0.0" / "files"
    if not cxr_files_path.exists():
        print(f"⚠️ CXR files directory not found at {cxr_files_path}. Skipping report load.")
        return {}

    print(f"Loading CXR reports from {cxr_files_path} ...")
    reports: Dict[int, str] = {}
    
    # Find all .txt files in the directory structure
    txt_files = list(cxr_files_path.glob("p*/p*/s*.txt"))
    print(f"  Found {len(txt_files):,} report files")
    
    for txt_file in tqdm(txt_files, desc="Reading CXR reports", unit="report"):
        # Extract study_id from filename (e.g., s51613820.txt -> 51613820)
        stem = txt_file.stem
        if stem.lower().startswith('s'):
            stem = stem[1:]
        try:
            study_id = int(stem)
        except ValueError:
            continue
        
        # Read the report text
        try:
            with open(txt_file, 'r', encoding='utf-8', errors='replace') as f:
                text = f.read().strip()
            reports[study_id] = text
        except Exception as e:
            print(f"Warning: Failed to read {txt_file}: {e}")
            continue

    if cache_path is not None:
        print(f"  Saving cached CXR reports to {cache_path}")
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with cache_path.open('wb') as f:
            pickle.dump(reports, f)

    print(f"  Loaded {len(reports):,} CXR reports.")
    return reports



In [25]:
def load_ecg_data(lazy: bool = False):
    """
    Load MIMIC-IV ECG dataset files.
    
    Args:
        lazy: If True, only load machine_measurements
        
    Returns:
        Tuple of dataframes: (patients, admissions, ecg_report, ecg_record_list)
    """
    print("Loading MIMIC-IV ECG dataset files...")
    
    ecg_path = MIMIC_IV_ECG_PATH / "1.0"
    
    print("  Loading ECG machine measurements...")
    ecg_report = pd.read_csv(ecg_path / "machine_measurements.csv")
    
    # Convert ecg_time to datetime
    ecg_report['ecg_time'] = pd.to_datetime(ecg_report['ecg_time'])
    
    if lazy:
        return None, None, ecg_report, None
    
    # Load patient and admission data from MIMIC-IV for matching
    mimiciv_path = MIMICIV_PATH / "2.2" / "hosp"
    print("  Loading patient data from MIMIC-IV...")
    patients = pd.read_csv(mimiciv_path / "patients.csv")
    patients.columns = patients.columns.str.lower()
    
    print("  Loading admissions data from MIMIC-IV...")
    admissions = pd.read_csv(mimiciv_path / "admissions.csv")
    admissions.columns = admissions.columns.str.lower()
    
    print("  Loading ECG record list...")
    ecg_record_list = pd.read_csv(ecg_path / "record_list.csv")
    
    print("✅ ECG data loaded successfully.")
    return patients, admissions, ecg_report, ecg_record_list


In [26]:
def load_notes_data(lazy: bool = False):
    """
    Load MIMIC-IV Notes dataset files.
    
    Args:
        lazy: If True, only load main note files (discharge, radiology)
        
    Returns:
        Tuple of dataframes: (discharge, radiology, discharge_detail, radiology_detail)
    """
    print("Loading MIMIC-IV Notes dataset files...")
    path = MIMIC_IV_NOTE_PATH / "2.2" / "note"
    
    print("  Loading discharge notes...")
    discharge = pd.read_csv(path / "discharge.csv")
    discharge.columns = discharge.columns.str.lower()
    discharge['charttime'] = pd.to_datetime(discharge['charttime'])
    discharge['storetime'] = pd.to_datetime(discharge['storetime'])
    
    print("  Loading radiology notes...")
    radiology = pd.read_csv(path / "radiology.csv")
    radiology.columns = radiology.columns.str.lower()
    radiology['charttime'] = pd.to_datetime(radiology['charttime'])
    radiology['storetime'] = pd.to_datetime(radiology['storetime'])
    
    if lazy:
        return discharge, radiology, None, None
    
    print("  Loading discharge detail...")
    discharge_detail = pd.read_csv(path / "discharge_detail.csv")
    discharge_detail.columns = discharge_detail.columns.str.lower()
    
    print("  Loading radiology detail...")
    radiology_detail = pd.read_csv(path / "radiology_detail.csv")
    radiology_detail.columns = radiology_detail.columns.str.lower()
    
    print("✅ Notes data loaded successfully.")
    return discharge, radiology, discharge_detail, radiology_detail


## Matching Algorithm

The matching algorithm links all modalities to admissions using temporal matching.


In [27]:
def match_modalities_to_admissions(
    admissions: pd.DataFrame,
    cxr_metadata: pd.DataFrame,
    ecg_report: pd.DataFrame,
    discharge_notes: pd.DataFrame,
    radiology_notes: pd.DataFrame,
    diagnoses: pd.DataFrame,
    cxr_reports_map: Optional[Dict[int, str]] = None,
    time_window_hours: int = 24,
    admission_time_slack_hours: int = 0,
    ecg_offsets_by_patient: Optional[Dict[int, Dict]] = None,
) -> List[Dict]:
    """
    Match all modalities to admissions.
    
    Args:
        admissions: EHR admissions dataframe
        cxr_metadata: CXR metadata dataframe
        ecg_report: ECG report dataframe
        discharge_notes: Discharge notes dataframe
        radiology_notes: Radiology notes dataframe
        diagnoses: Diagnoses dataframe
        cxr_reports_map: Optional mapping of CXR study_id to final report text
        time_window_hours: Time window for radiology note matching to CXR (default 24 hours)
        admission_time_slack_hours: Extend admission window on both sides to allow for minor clock misalignments
        
    Returns:
        List of dictionaries, each representing a matched admission record
    """
    print("=" * 80)
    print("Matching Modalities to Admissions")
    print("=" * 80)
    
    # Create index for faster lookups
    print("\nCreating indices for faster lookups...")
    cxr_by_subject = cxr_metadata.groupby('subject_id')
    ecg_by_subject = ecg_report.groupby('subject_id')
    radiology_by_subject = radiology_notes.groupby('subject_id')
    diagnoses_by_admission = diagnoses.groupby('hadm_id')
    
    # Filter radiology notes for chest-related (for CXR matching)
    chest_radiology = radiology_notes[
        radiology_notes['text'].str.contains('CHEST|CXR|CHEST X', case=False, na=False)
    ].copy()
    chest_radiology_by_subject = chest_radiology.groupby('subject_id')
    
    matched_records = []

    # Admission time slack for cross-dataset misalignment
    slack_td = timedelta(hours=admission_time_slack_hours) if admission_time_slack_hours and admission_time_slack_hours > 0 else timedelta(0)
    
    print(f"\nProcessing {len(admissions)} admissions...")
    for idx, adm in tqdm(admissions.iterrows(), total=len(admissions)):
        subject_id = adm['subject_id']
        hadm_id = adm['hadm_id']
        admittime = adm['admittime']
        dischtime = adm['dischtime']
        window_start = admittime - slack_td
        window_end = dischtime + slack_td
        
        # Initialize record
        record = {
            'subject_id': subject_id,
            'hadm_id': hadm_id,
            'admittime': admittime,
            'dischtime': dischtime,
            'admission_type': adm.get('admission_type', None),
            'admission_location': adm.get('admission_location', None),
            'discharge_location': adm.get('discharge_location', None),
            'has_ehr': False,
            'has_cxr': False,
            'has_ecg': False,
            'has_discharge_note': False,
            'has_radiology_note': False,
            'diagnoses': [],
            'cxr_studies': [],
            'ecg_records': [],
            'discharge_note': None,
            'radiology_notes': []
        }
        
        # 1. Link EHR Data (Diagnoses) - Direct link via hadm_id
        if hadm_id in diagnoses_by_admission.groups:
            patient_diag = diagnoses_by_admission.get_group(hadm_id)
            record['diagnoses'] = patient_diag.to_dict('records')
            record['has_ehr'] = True
        
        # 2. Link Discharge Notes - Direct link via hadm_id
        discharge_note = discharge_notes[discharge_notes['hadm_id'] == hadm_id]
        if len(discharge_note) > 0:
            record['discharge_note'] = discharge_note.iloc[0].to_dict()
            record['has_discharge_note'] = True
        
        # 3. Match CXR Studies - Temporal matching
        if subject_id in cxr_by_subject.groups:
            patient_cxr = cxr_by_subject.get_group(subject_id)
            matching_cxr = patient_cxr[
                (patient_cxr['study_datetime'] >= window_start) & 
                (patient_cxr['study_datetime'] <= window_end)
            ]
            
            if len(matching_cxr) > 0:
                # Group by study_id
                for study_id in matching_cxr['study_id'].unique():
                    study_cxr = matching_cxr[matching_cxr['study_id'] == study_id]
                    study_record = {
                        'study_id': int(study_id),
                        'images': []
                    }
                    if cxr_reports_map:
                        report_text = cxr_reports_map.get(int(study_id))
                        if report_text is not None:
                            study_record['report_text'] = report_text
                    for _, cxr in study_cxr.iterrows():
                        study_record['images'].append({
                            'dicom_id': cxr['dicom_id'],
                            'study_datetime': cxr['study_datetime'].isoformat() if pd.notna(cxr['study_datetime']) else None,
                            'view_position': cxr.get('ViewPosition', None)
                        })
                    record['cxr_studies'].append(study_record)
                record['has_cxr'] = True
        
        # 4. Match ECG Records - Temporal matching
        if subject_id in ecg_by_subject.groups:
            patient_ecg = ecg_by_subject.get_group(subject_id).copy()
            # Apply per-patient offset for comparison if available
            offset_hours = 0
            if ecg_offsets_by_patient and subject_id in ecg_offsets_by_patient:
                offset_hours = int(ecg_offsets_by_patient[subject_id].get('offset_hours', 0) or 0)
            if offset_hours != 0:
                patient_ecg['ecg_time_for_match'] = patient_ecg['ecg_time'] + pd.to_timedelta(offset_hours, unit='h')
            else:
                patient_ecg['ecg_time_for_match'] = patient_ecg['ecg_time']

            matching_ecg = patient_ecg[
                (patient_ecg['ecg_time_for_match'] >= window_start) & 
                (patient_ecg['ecg_time_for_match'] <= window_end)
            ]
            
            if len(matching_ecg) > 0:
                for _, ecg in matching_ecg.iterrows():
                    # Combine report fields
                    report_parts = []
                    for i in range(18):
                        val = ecg.get(f'report_{i}', None)
                        if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
                            report_parts.append(str(val).strip())
                    full_report = ', '.join(report_parts) if report_parts else None
                    
                    ecg_record = {
                        'study_id': int(ecg['study_id']),
                        'cart_id': int(ecg['cart_id']) if pd.notna(ecg.get('cart_id')) else None,
                        'ecg_time': ecg['ecg_time'].isoformat() if pd.notna(ecg['ecg_time']) else None,
                        'ecg_time_matched': ecg['ecg_time_for_match'].isoformat() if pd.notna(ecg['ecg_time_for_match']) else None,
                        'time_offset_hours': offset_hours if offset_hours != 0 else None,
                        'report': full_report,
                        'rr_interval': float(ecg['rr_interval']) if pd.notna(ecg.get('rr_interval')) else None,
                        'p_axis': float(ecg['p_axis']) if pd.notna(ecg.get('p_axis')) else None,
                        'qrs_axis': float(ecg['qrs_axis']) if pd.notna(ecg.get('qrs_axis')) else None,
                        't_axis': float(ecg['t_axis']) if pd.notna(ecg.get('t_axis')) else None
                    }
                    record['ecg_records'].append(ecg_record)
                record['has_ecg'] = True
        
        # 5. Match Radiology Notes - Temporal matching (for chest-related notes)
        if subject_id in chest_radiology_by_subject.groups:
            patient_rad = chest_radiology_by_subject.get_group(subject_id)
            matching_rad = patient_rad[
                (patient_rad['charttime'].fillna(patient_rad['storetime']) >= window_start) & 
                (patient_rad['charttime'].fillna(patient_rad['storetime']) <= window_end)
            ]
            
            if len(matching_rad) > 0:
                for _, rad in matching_rad.iterrows():
                    rad_time = rad['charttime'] if pd.notna(rad['charttime']) else rad['storetime']
                    # Try to match to CXR studies within time window
                    matched_cxr_study = None
                    if len(record['cxr_studies']) > 0:
                        for cxr_study in record['cxr_studies']:
                            # Find first image in study to get time
                            if len(cxr_study['images']) > 0:
                                cxr_time_str = cxr_study['images'][0].get('study_datetime')
                                if cxr_time_str:
                                    cxr_time = pd.to_datetime(cxr_time_str)
                                    if abs((rad_time - cxr_time).total_seconds()) <= time_window_hours * 3600:
                                        matched_cxr_study = cxr_study['study_id']
                                        break
                    
                    rad_record = {
                        'note_id': rad['note_id'],
                        'charttime': rad['charttime'].isoformat() if pd.notna(rad['charttime']) else None,
                        'storetime': rad['storetime'].isoformat() if pd.notna(rad['storetime']) else None,
                        'matched_cxr_study_id': matched_cxr_study,
                        'text_preview': rad['text'][:200] + '...' if pd.notna(rad.get('text')) and len(str(rad.get('text', ''))) > 200 else rad.get('text', None)
                    }
                    record['radiology_notes'].append(rad_record)
                record['has_radiology_note'] = True
        
        matched_records.append(record)
    
    print(f"\n✅ Matching complete! Processed {len(matched_records)} admissions.")
    return matched_records


In [28]:
def compute_statistics(matched_records: List[Dict]) -> Dict:
    """
    Compute statistics on matched records.
    
    Args:
        matched_records: List of matched admission records
        
    Returns:
        Dictionary containing statistics
    """
    print("=" * 80)
    print("Computing Statistics")
    print("=" * 80)
    
    stats = {
        'total_admissions': len(matched_records),
        'modality_counts': {},
        'modality_combinations': defaultdict(int),
        'sequence_lengths': {
            'by_patient': defaultdict(int),
            'admissions_per_patient': []
        },
        'modality_details': {
            'cxr': {'total_studies': 0, 'total_images': 0, 'admissions_with_cxr': 0},
            'ecg': {'total_records': 0, 'admissions_with_ecg': 0},
            'ehr': {'total_diagnoses': 0, 'admissions_with_ehr': 0},
            'discharge_note': {'admissions_with_note': 0},
            'radiology_note': {'total_notes': 0, 'admissions_with_note': 0}
        }
    }
    
    # Count modalities
    modality_flags = ['has_ehr', 'has_cxr', 'has_ecg', 'has_discharge_note', 'has_radiology_note']
    for flag in modality_flags:
        stats['modality_counts'][flag] = sum(1 for r in matched_records if r.get(flag, False))
    
    # Count modality combinations
    for record in matched_records:
        combo = []
        if record.get('has_ehr', False):
            combo.append('EHR')
        if record.get('has_cxr', False):
            combo.append('CXR')
        if record.get('has_ecg', False):
            combo.append('ECG')
        if record.get('has_discharge_note', False):
            combo.append('Discharge')
        if record.get('has_radiology_note', False):
            combo.append('Radiology')
        
        if combo:
            combo_str = '+'.join(sorted(combo))
            stats['modality_combinations'][combo_str] += 1
        else:
            stats['modality_combinations']['None'] += 1
    
    # Count sequence lengths (admissions per patient)
    patient_admissions = defaultdict(list)
    for record in matched_records:
        patient_admissions[record['subject_id']].append(record)
    
    for patient_id, admissions in patient_admissions.items():
        num_admissions = len(admissions)
        stats['sequence_lengths']['admissions_per_patient'].append(num_admissions)
        stats['sequence_lengths']['by_patient'][num_admissions] += 1
    
    # Detailed modality statistics
    for record in matched_records:
        if record.get('has_ehr', False):
            stats['modality_details']['ehr']['admissions_with_ehr'] += 1
            stats['modality_details']['ehr']['total_diagnoses'] += len(record.get('diagnoses', []))
        
        if record.get('has_cxr', False):
            stats['modality_details']['cxr']['admissions_with_cxr'] += 1
            for study in record.get('cxr_studies', []):
                stats['modality_details']['cxr']['total_studies'] += 1
                stats['modality_details']['cxr']['total_images'] += len(study.get('images', []))
        
        if record.get('has_ecg', False):
            stats['modality_details']['ecg']['admissions_with_ecg'] += 1
            stats['modality_details']['ecg']['total_records'] += len(record.get('ecg_records', []))
        
        if record.get('has_discharge_note', False):
            stats['modality_details']['discharge_note']['admissions_with_note'] += 1
        
        if record.get('has_radiology_note', False):
            stats['modality_details']['radiology_note']['admissions_with_note'] += 1
            stats['modality_details']['radiology_note']['total_notes'] += len(record.get('radiology_notes', []))
    
    # Convert defaultdicts to regular dicts for JSON serialization
    stats['modality_combinations'] = dict(stats['modality_combinations'])
    stats['sequence_lengths']['by_patient'] = dict(stats['sequence_lengths']['by_patient'])
    
    return stats


def print_statistics(stats: Dict, matched_records: Optional[List[Dict]] = None):
    """
    Print statistics in a readable format.
    
    Args:
        stats: Statistics dictionary from compute_statistics
        matched_records: Optional list of matched records (for unique patient count)
    """
    print("\n" + "=" * 80)
    print("MATCHING STATISTICS")
    print("=" * 80)
    
    print(f"\n📊 Total Admissions: {stats['total_admissions']:,}")
    if matched_records is not None:
        print(f"📊 Unique Patients: {len(set([r['subject_id'] for r in matched_records])):,}")
    
    print("\n" + "-" * 80)
    print("Modality Presence:")
    print("-" * 80)
    for flag, count in stats['modality_counts'].items():
        percentage = (count / stats['total_admissions']) * 100
        modality_name = flag.replace('has_', '').replace('_', ' ').title()
        print(f"  {modality_name:20s}: {count:8,} ({percentage:5.2f}%)")
    
    print("\n" + "-" * 80)
    print("Modality Combinations (Top 10):")
    print("-" * 80)
    sorted_combos = sorted(stats['modality_combinations'].items(), key=lambda x: x[1], reverse=True)
    for combo, count in sorted_combos[:10]:
        percentage = (count / stats['total_admissions']) * 100
        print(f"  {combo:30s}: {count:8,} ({percentage:5.2f}%)")
    
    print("\n" + "-" * 80)
    print("Sequence Lengths (Admissions per Patient):")
    print("-" * 80)
    admissions_per_patient = stats['sequence_lengths']['admissions_per_patient']
    if admissions_per_patient:
        print(f"  Mean:  {np.mean(admissions_per_patient):.2f}")
        print(f"  Median: {np.median(admissions_per_patient):.2f}")
        print(f"  Min:   {np.min(admissions_per_patient)}")
        print(f"  Max:   {np.max(admissions_per_patient)}")
        print(f"\n  Distribution:")
        for num_adm, count in sorted(stats['sequence_lengths']['by_patient'].items())[:15]:
            print(f"    {num_adm} admission(s): {count:,} patients")
    
    print("\n" + "-" * 80)
    print("Modality Details:")
    print("-" * 80)
    details = stats['modality_details']
    print(f"\n  EHR:")
    print(f"    Admissions with EHR: {details['ehr']['admissions_with_ehr']:,}")
    print(f"    Total diagnoses: {details['ehr']['total_diagnoses']:,}")
    
    print(f"\n  CXR:")
    print(f"    Admissions with CXR: {details['cxr']['admissions_with_cxr']:,}")
    print(f"    Total studies: {details['cxr']['total_studies']:,}")
    print(f"    Total images: {details['cxr']['total_images']:,}")
    
    print(f"\n  ECG:")
    print(f"    Admissions with ECG: {details['ecg']['admissions_with_ecg']:,}")
    print(f"    Total records: {details['ecg']['total_records']:,}")
    
    print(f"\n  Notes:")
    print(f"    Admissions with discharge note: {details['discharge_note']['admissions_with_note']:,}")
    print(f"    Admissions with radiology note: {details['radiology_note']['admissions_with_note']:,}")
    print(f"    Total radiology notes: {details['radiology_note']['total_notes']:,}")
    
    print("\n" + "=" * 80)


In [29]:
def _safe_parse_time(value):
    if value is None:
        return pd.NaT
    try:
        return pd.to_datetime(value)
    except Exception:
        return pd.NaT


def sort_matched_records(matched_records: List[Dict]) -> List[Dict]:
    """
    Return a new list where:
    - Records are sorted by (subject_id, admittime, hadm_id)
    - Within each record:
      * CXR images are sorted by study_datetime
      * CXR studies are sorted by the earliest image time
      * ECG records are sorted by ecg_time
      * Radiology notes are sorted by charttime/storetime
    """
    sorted_records: List[Dict] = []

    for rec in matched_records:
        rec = dict(rec)  # shallow copy

        # Sort CXR images within each study
        cxr_studies = rec.get("cxr_studies", []) or []
        for study in cxr_studies:
            images = study.get("images", []) or []
            images = sorted(
                images,
                key=lambda img: _safe_parse_time(img.get("study_datetime"))
            )
            study["images"] = images

        # Sort studies by earliest image time
        cxr_studies = sorted(
            cxr_studies,
            key=lambda st: _safe_parse_time(
                (st.get("images") or [{}])[0].get("study_datetime") if st.get("images") else None
            )
        )
        rec["cxr_studies"] = cxr_studies

        # Sort ECG records
        ecg_records = rec.get("ecg_records", []) or []
        ecg_records = sorted(ecg_records, key=lambda e: _safe_parse_time(e.get("ecg_time")))
        rec["ecg_records"] = ecg_records

        # Sort radiology notes
        rad_notes = rec.get("radiology_notes", []) or []
        def _rad_time(r):
            t = r.get("charttime") or r.get("storetime")
            return _safe_parse_time(t)
        rad_notes = sorted(rad_notes, key=_rad_time)
        rec["radiology_notes"] = rad_notes

        sorted_records.append(rec)

    # Global ordering by (subject_id, admittime, hadm_id)
    def _adm_time(r):
        return _safe_parse_time(r.get("admittime"))

    def _hadm_key(r):
        hadm = r.get("hadm_id", -1)
        try:
            return int(hadm)
        except Exception:
            return -1

    sorted_records = sorted(
        sorted_records,
        key=lambda r: (r.get("subject_id", -1), _adm_time(r), _hadm_key(r))
    )
    return sorted_records



In [30]:
def estimate_ecg_time_offsets(
    admissions: pd.DataFrame,
    ecg_report: pd.DataFrame,
    subject_ids: Optional[List[int]] = None,
    grid_min_hours: int = -168,
    grid_max_hours: int = 168,
    grid_step_hours: int = 6,
    min_lift_events: int = 5,
    min_relative_lift: float = 0.25,
    max_abs_hours_accept: int = 168,
    admission_time_slack_hours: int = 6,
) -> Dict[int, Dict]:
    """
    Estimate per-patient ECG time offset that maximizes overlap of ECG times with
    admission windows. Returns dict: subject_id -> {
        'offset_hours': int,
        'lift': int,
        'baseline_in': int,
        'shifted_in': int,
        'n_events': int,
        'confidence': float
    }
    Only offsets passing acceptance criteria are returned.
    """
    slack_td = timedelta(hours=admission_time_slack_hours)

    # Build windows per subject
    windows_by_subject: Dict[int, List[Tuple[pd.Timestamp, pd.Timestamp]]] = defaultdict(list)
    for _, adm in admissions.iterrows():
        windows_by_subject[int(adm['subject_id'])].append(
            (adm['admittime'] - slack_td, adm['dischtime'] + slack_td)
        )

    def count_in_windows(times: np.ndarray, windows: List[Tuple[pd.Timestamp, pd.Timestamp]]) -> int:
        if len(windows) == 0 or times.size == 0:
            return 0
        count = 0
        for t in times:
            if pd.isna(t):
                continue
            for s, e in windows:
                if s <= t <= e:
                    count += 1
                    break
        return count

    ecg_by_subject = ecg_report.groupby('subject_id')
    results: Dict[int, Dict] = {}

    pid_iter = subject_ids if subject_ids is not None else list(ecg_by_subject.groups.keys())
    for pid in pid_iter:
        if pid not in ecg_by_subject.groups:
            continue
        windows = windows_by_subject.get(int(pid), [])
        ecg_df = ecg_by_subject.get_group(pid)
        times = ecg_df['ecg_time'].to_numpy()
        n_events = times.size
        if n_events == 0:
            continue

        baseline_in = count_in_windows(times, windows)
        best_offset = 0
        best_in = baseline_in

        for off in range(grid_min_hours, grid_max_hours + 1, grid_step_hours):
            if off == 0:
                continue
            shifted = times + np.array([np.timedelta64(off, 'h')] * n_events)
            in_cnt = count_in_windows(shifted, windows)
            if in_cnt > best_in:
                best_in = in_cnt
                best_offset = off

        lift = best_in - baseline_in
        rel = (lift / max(1, baseline_in)) if baseline_in > 0 else (best_in / max(1, n_events))
        abs_ok = abs(best_offset) <= max_abs_hours_accept
        accept = (lift >= min_lift_events) and (rel >= min_relative_lift) and abs_ok

        if accept and best_offset != 0:
            confidence = best_in / max(1, n_events)
            results[int(pid)] = {
                'offset_hours': int(best_offset),
                'lift': int(lift),
                'baseline_in': int(baseline_in),
                'shifted_in': int(best_in),
                'n_events': int(n_events),
                'confidence': float(confidence)
            }

    return results


In [31]:
def collect_unassigned_events(
    admissions: pd.DataFrame,
    cxr_metadata: pd.DataFrame,
    ecg_report: pd.DataFrame,
    radiology_notes: pd.DataFrame,
    matched_records: List[Dict],
    cxr_reports_map: Optional[Dict[int, str]] = None,
    admission_time_slack_hours: int = 0,
    drop_cxr_beyond_days: Optional[int] = 90,
) -> Dict[int, Dict[str, List[Dict]]]:
    """
    For each patient, collect modality events (CXR studies, ECG records, radiology notes)
    that do NOT fall into any admission time window (with slack).

    Returns a dict: subject_id -> { 'cxr_studies': [...], 'ecg_records': [...], 'radiology_notes': [...] }
    """
    slack_td = timedelta(hours=admission_time_slack_hours) if admission_time_slack_hours and admission_time_slack_hours > 0 else timedelta(0)

    # Build admission windows per subject
    windows_by_subject: Dict[int, List[Tuple[pd.Timestamp, pd.Timestamp]]] = defaultdict(list)
    for _, adm in admissions.iterrows():
        pid = adm['subject_id']
        start = adm['admittime'] - slack_td
        end = adm['dischtime'] + slack_td
        windows_by_subject[pid].append((start, end))

    # Helper: check if a time falls in any window for subject
    def in_any_window(pid: int, t: pd.Timestamp) -> bool:
        if pd.isna(t):
            return False
        for (s, e) in windows_by_subject.get(pid, []):
            if s <= t <= e:
                return True
        return False

    # Filter chest-related radiology like in matching
    chest_radiology = radiology_notes[
        radiology_notes['text'].str.contains('CHEST|CXR|CHEST X', case=False, na=False)
    ].copy()

    unassigned: Dict[int, Dict[str, List[Dict]]] = defaultdict(lambda: {'cxr_studies': [], 'ecg_records': [], 'radiology_notes': []})

    # CXR: work at study level using earliest image time
    if cxr_metadata is not None and not cxr_metadata.empty:
        for pid, df in cxr_metadata.groupby('subject_id'):
            # compute earliest time per study
            df_tmp = df.copy()
            df_tmp['study_time'] = df_tmp['study_datetime']
            for study_id, sub in df_tmp.groupby('study_id'):
                t = pd.to_datetime(sub['study_time'].dropna().min()) if sub['study_time'].notna().any() else pd.NaT
                if not in_any_window(pid, t):
                    # Optionally drop way-off CXR far from any admission window
                    if drop_cxr_beyond_days is not None and len(windows_by_subject.get(pid, [])) > 0 and pd.notna(t):
                        # compute minimum distance to any window
                        min_dist_days = None
                        for (s, e) in windows_by_subject[pid]:
                            if t < s:
                                days = (s - t).total_seconds() / 86400.0
                            elif t > e:
                                days = (t - e).total_seconds() / 86400.0
                            else:
                                days = 0.0
                            min_dist_days = days if min_dist_days is None else min(min_dist_days, days)
                        if min_dist_days is not None and min_dist_days > float(drop_cxr_beyond_days):
                            continue
                    # collect all images like in matching for symmetry
                    images = []
                    for _, row in sub.iterrows():
                        images.append({
                            'dicom_id': row['dicom_id'],
                            'study_datetime': row['study_datetime'].isoformat() if pd.notna(row['study_datetime']) else None,
                            'view_position': row.get('ViewPosition', None)
                        })
                    cxr_entry = {'study_id': int(study_id), 'images': images}
                    if cxr_reports_map:
                        report_text = cxr_reports_map.get(int(study_id))
                        if report_text is not None:
                            cxr_entry['report_text'] = report_text
                    unassigned[pid]['cxr_studies'].append(cxr_entry)

    # ECG: individual records by ecg_time
    if ecg_report is not None and not ecg_report.empty:
        for pid, df in ecg_report.groupby('subject_id'):
            for _, ecg in df.iterrows():
                t = ecg['ecg_time']
                if not in_any_window(pid, t):
                    report_parts = []
                    for i in range(18):
                        val = ecg.get(f'report_{i}', None)
                        if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
                            report_parts.append(str(val).strip())
                    full_report = ', '.join(report_parts) if report_parts else None
                    unassigned[pid]['ecg_records'].append({
                        'study_id': int(ecg['study_id']),
                        'cart_id': int(ecg['cart_id']) if pd.notna(ecg.get('cart_id')) else None,
                        'ecg_time': ecg['ecg_time'].isoformat() if pd.notna(ecg['ecg_time']) else None,
                        'report': full_report,
                        'rr_interval': float(ecg['rr_interval']) if pd.notna(ecg.get('rr_interval')) else None,
                        'p_axis': float(ecg['p_axis']) if pd.notna(ecg.get('p_axis')) else None,
                        'qrs_axis': float(ecg['qrs_axis']) if pd.notna(ecg.get('qrs_axis')) else None,
                        't_axis': float(ecg['t_axis']) if pd.notna(ecg.get('t_axis')) else None
                    })

    # Radiology notes: chest-only, by charttime/storetime
    if chest_radiology is not None and not chest_radiology.empty:
        for pid, df in chest_radiology.groupby('subject_id'):
            for _, rad in df.iterrows():
                t = rad['charttime'] if pd.notna(rad['charttime']) else rad['storetime']
                if not in_any_window(pid, t):
                    unassigned[pid]['radiology_notes'].append({
                        'note_id': rad['note_id'],
                        'charttime': rad['charttime'].isoformat() if pd.notna(rad['charttime']) else None,
                        'storetime': rad['storetime'].isoformat() if pd.notna(rad['storetime']) else None,
                        'text_preview': rad['text'][:200] + '...' if pd.notna(rad.get('text')) and len(str(rad.get('text', ''))) > 200 else rad.get('text', None)
                    })

    # Remove empty entries
    result: Dict[int, Dict[str, List[Dict]]] = {}
    for pid, payload in unassigned.items():
        if payload['cxr_studies'] or payload['ecg_records'] or payload['radiology_notes']:
            result[pid] = payload

    return result


In [32]:
from dateutil.relativedelta import relativedelta

def _extract_event_time(event: Dict) -> Optional[pd.Timestamp]:
    t = None
    if 'ecg_time' in event and event.get('ecg_time'):
        t = pd.to_datetime(event['ecg_time'], errors='coerce')
    elif 'ecg_time_matched' in event and event.get('ecg_time_matched'):
        t = pd.to_datetime(event['ecg_time_matched'], errors='coerce')
    elif 'charttime' in event and event.get('charttime'):
        t = pd.to_datetime(event['charttime'], errors='coerce')
    elif 'storetime' in event and event.get('storetime'):
        t = pd.to_datetime(event['storetime'], errors='coerce')
    return t


def _extract_cxr_study_time(study: Dict) -> Optional[pd.Timestamp]:
    imgs = study.get('images') or []
    if not imgs:
        return None
    t = imgs[0].get('study_datetime')
    return pd.to_datetime(t, errors='coerce') if t else None


def build_virtual_visits_from_unassigned(
    matched_records: List[Dict],
    unassigned_by_patient: Dict[int, Dict[str, List[Dict]]],
    max_virtual_per_gap: int = 4,
    max_virtual_pre: int = 3,
    max_virtual_post: int = 3,
    min_events_per_virtual: int = 2,
    bin_level_sequence: List[str] = None,
) -> List[Dict]:
    """
    Create virtual visits from unassigned events and return augmented records (real + virtual).
    Binning strategy: month-level bins by default, with adaptive merging to satisfy caps.
    """
    if bin_level_sequence is None:
        bin_level_sequence = ['month']

    # Index real admissions by patient and sort
    admissions_by_patient: Dict[int, List[Dict]] = defaultdict(list)
    for rec in matched_records:
        admissions_by_patient[rec['subject_id']].append(rec)
    for pid in admissions_by_patient:
        admissions_by_patient[pid] = sorted(admissions_by_patient[pid], key=lambda r: r['admittime'])

    augmented: List[Dict] = list(matched_records)

    virtual_hadm_seed = -1000000000
    virtual_counter = 0

    for pid, payload in unassigned_by_patient.items():
        real_list = admissions_by_patient.get(pid, [])
        if not payload or (not payload.get('cxr_studies') and not payload.get('ecg_records') and not payload.get('radiology_notes')):
            continue

        # Build timeline segments
        anchors = []
        for rec in real_list:
            anchors.append(('adm_start', rec['admittime']))
            anchors.append(('adm_end', rec['dischtime']))
        anchors = sorted(anchors, key=lambda x: x[1])

        # Collect all events with times
        events = []
        for st in payload.get('cxr_studies', []):
            t = _extract_cxr_study_time(st)
            if pd.notna(t):
                events.append(('cxr', t, st))
        for e in payload.get('ecg_records', []):
            t = _extract_event_time(e)
            if pd.notna(t):
                events.append(('ecg', t, e))
        for r in payload.get('radiology_notes', []):
            t = _extract_event_time(r)
            if pd.notna(t):
                events.append(('rad', t, r))
        if not events:
            continue
        events.sort(key=lambda x: x[1])

        # Define segments: pre, between, post
        segs = []
        if real_list:
            first_start = real_list[0]['admittime']
            last_end = real_list[-1]['dischtime']
            segs.append(('pre', pd.Timestamp.min, first_start))
            for i in range(len(real_list) - 1):
                segs.append(('between', real_list[i]['dischtime'], real_list[i+1]['admittime']))
            segs.append(('post', last_end, pd.Timestamp.max))
        else:
            segs.append(('pre', pd.Timestamp.min, pd.Timestamp.max))

        # Assign events to segments
        seg_events: Dict[Tuple[str, int], List[Tuple[str, pd.Timestamp, Dict]]] = defaultdict(list)
        for kind, t, obj in events:
            for idx, (stype, s_start, s_end) in enumerate(segs):
                if (t >= s_start) and (t <= s_end):
                    seg_events[(stype, idx)].append((kind, t, obj))
                    break

        # For each segment, bin by month and create virtual visits, enforce caps
        for (stype, idx), evs in seg_events.items():
            if not evs:
                continue
            # Monthly bin
            bins: Dict[Tuple[int, int], List[Tuple[str, pd.Timestamp, Dict]]] = defaultdict(list)
            for kind, t, obj in evs:
                bins[(t.year, t.month)].append((kind, t, obj))
            # Sort bins chronologically
            bin_keys = sorted(bins.keys())

            # Enforce caps
            if stype == 'pre':
                cap = max_virtual_pre
            elif stype == 'post':
                cap = max_virtual_post
            else:
                cap = max_virtual_per_gap

            # Merge bins if exceeding cap
            if cap is not None and len(bin_keys) > cap:
                # Merge into cap chunks preserving order
                chunk_size = int(np.ceil(len(bin_keys) / cap))
                merged_keys = []
                merged_bins: List[List[Tuple[str, pd.Timestamp, Dict]]] = []
                for i in range(0, len(bin_keys), chunk_size):
                    group_keys = bin_keys[i:i+chunk_size]
                    merged_keys.append((group_keys[0], group_keys[-1]))
                    merged_list = []
                    for k in group_keys:
                        merged_list.extend(bins[k])
                    merged_bins.append(merged_list)
                # Replace bins with merged
                bins = {}
                bin_keys = []
                for i, m_evs in enumerate(merged_bins):
                    bins[(i,)] = m_evs
                    bin_keys.append((i,))

            # Create virtual visits per bin
            vseq = 0
            for bkey in bin_keys:
                bevs = bins[bkey]
                # Filter by min events
                if len(bevs) < min_events_per_virtual:
                    continue
                times = [t for _, t, _ in bevs]
                bin_start = min(times)
                # Use end as end of that month or max time in bevs
                if isinstance(bkey, tuple) and len(bkey) == 2:
                    y, m = bkey
                    bin_end = (pd.Timestamp(year=y, month=m, day=1) + relativedelta(months=1)) - pd.Timedelta(seconds=1)
                else:
                    bin_end = max(times)
                # Use pandas median on timestamps to avoid numpy datetime ufunc issues
                times_ts = pd.to_datetime(times)
                anchor = pd.to_datetime(pd.Series(times_ts).median())

                # Aggregate payloads
                cxr_studies = [obj for kind, _, obj in bevs if kind == 'cxr']
                ecg_records = [obj for kind, _, obj in bevs if kind == 'ecg']
                rad_notes = [obj for kind, _, obj in bevs if kind == 'rad']

                # Compose virtual record
                vseq += 1
                virtual_counter += 1
                synthetic_hadm_id = virtual_hadm_seed - virtual_counter
                virtual_id = f"V-{pid}-{stype}-{idx}-{vseq}"
                virtual_record = {
                    'subject_id': pid,
                    'hadm_id': synthetic_hadm_id,
                    'admittime': bin_start,
                    'dischtime': bin_end,
                    'admission_type': None,
                    'admission_location': None,
                    'discharge_location': None,
                    'has_ehr': False,
                    'has_cxr': len(cxr_studies) > 0,
                    'has_ecg': len(ecg_records) > 0,
                    'has_discharge_note': False,
                    'has_radiology_note': len(rad_notes) > 0,
                    'diagnoses': [],
                    'cxr_studies': cxr_studies,
                    'ecg_records': ecg_records,
                    'discharge_note': None,
                    'radiology_notes': rad_notes,
                    'is_virtual': True,
                    'virtual_group': stype,
                    'bin_size': 'month',
                    'bin_start': bin_start.isoformat() if pd.notna(bin_start) else None,
                    'bin_end': bin_end.isoformat() if pd.notna(bin_end) else None,
                    'anchor_time': anchor.isoformat() if pd.notna(anchor) else None,
                    'virtual_id': virtual_id,
                }
                augmented.append(virtual_record)

    # Sort augmented records deterministically
    augmented = sort_matched_records(augmented)
    return augmented



In [33]:
def display_sample_patient(matched_records: List[Dict], subject_id: Optional[int] = None, 
                            min_modalities: int = 2, max_admissions: int = 5):
    """
    Display a sample patient record with multiple modalities.
    
    Args:
        matched_records: List of matched admission records
        subject_id: Specific subject_id to display (if None, picks a patient with >= min_modalities)
        min_modalities: Minimum number of modalities required (default: 2)
        max_admissions: Maximum number of admissions to display (default: 5)
    """
    print("=" * 80)
    print("SAMPLE PATIENT RECORD")
    print("=" * 80)
    
    # Find patient records
    if subject_id is None:
        # Find a patient with multiple modalities
        patient_admissions = defaultdict(list)
        for record in matched_records:
            patient_admissions[record['subject_id']].append(record)
        
        # Find patient with at least min_modalities
        for pid, admissions in patient_admissions.items():
            # Count unique modalities across all admissions
            modalities = set()
            for adm in admissions:
                if adm.get('has_ehr', False):
                    modalities.add('EHR')
                if adm.get('has_cxr', False):
                    modalities.add('CXR')
                if adm.get('has_ecg', False):
                    modalities.add('ECG')
                if adm.get('has_discharge_note', False):
                    modalities.add('Discharge')
                if adm.get('has_radiology_note', False):
                    modalities.add('Radiology')
            
            if len(modalities) >= min_modalities:
                subject_id = pid
                break
        
        if subject_id is None:
            print(f"\n⚠️  No patient found with {min_modalities}+ modalities. Showing first patient...")
            subject_id = matched_records[0]['subject_id']
    
    # Get all admissions for this patient
    patient_records = [r for r in matched_records if r['subject_id'] == subject_id]
    patient_records = sorted(patient_records, key=lambda x: x['admittime'])[:max_admissions]
    
    print(f"\nPatient ID: {subject_id}")
    print(f"Total Admissions: {len([r for r in matched_records if r['subject_id'] == subject_id])}")
    print(f"Displaying: {len(patient_records)} admission(s)")
    
    for idx, record in enumerate(patient_records, 1):
        print(f"\n{'=' * 80}")
        print(f"ADMISSION #{idx}: HADM ID {record['hadm_id']}")
        print(f"{'=' * 80}")
        
        print(f"\n📅 Admission Period:")
        print(f"   Admit:    {record['admittime']}")
        print(f"   Discharge: {record['dischtime']}")
        print(f"   Duration: {record['dischtime'] - record['admittime']}")
        if record.get('admission_type'):
            print(f"   Type:     {record['admission_type']}")
        
        # Modality flags
        modalities = []
        if record.get('has_ehr', False):
            modalities.append('EHR')
        if record.get('has_cxr', False):
            modalities.append('CXR')
        if record.get('has_ecg', False):
            modalities.append('ECG')
        if record.get('has_discharge_note', False):
            modalities.append('Discharge Note')
        if record.get('has_radiology_note', False):
            modalities.append('Radiology Note')
        
        print(f"\n📋 Modalities: {', '.join(modalities) if modalities else 'None'}")
        
        # EHR Data
        if record.get('has_ehr', False):
            diagnoses = record.get('diagnoses', [])
            print(f"\n🏥 EHR Data:")
            print(f"   Diagnoses: {len(diagnoses)}")
            for diag in diagnoses[:5]:  # Show first 5
                icd_code = diag.get('icd_code', 'N/A')
                icd_version = diag.get('icd_version', 'N/A')
                seq = diag.get('seq_num', 'N/A')
                print(f"     [{seq}] ICD-{icd_version}: {icd_code}")
            if len(diagnoses) > 5:
                print(f"     ... and {len(diagnoses) - 5} more")
        
        # CXR Data
        if record.get('has_cxr', False):
            cxr_studies = record.get('cxr_studies', [])
            print(f"\n📷 CXR Data:")
            print(f"   Studies: {len(cxr_studies)}")
            total_images = sum(len(s.get('images', [])) for s in cxr_studies)
            print(f"   Total Images: {total_images}")
            for study in cxr_studies[:2]:  # Show first 2 studies
                print(f"     Study ID: {study['study_id']}")
                print(f"       Images: {len(study.get('images', []))}")
                if study.get('images'):
                    img = study['images'][0]
                    print(f"       First image: {img.get('dicom_id', 'N/A')[:40]}...")
                    print(f"       Time: {img.get('study_datetime', 'N/A')}")
        
        # ECG Data
        if record.get('has_ecg', False):
            ecg_records = record.get('ecg_records', [])
            print(f"\n📊 ECG Data:")
            print(f"   Records: {len(ecg_records)}")
            for ecg in ecg_records[:2]:  # Show first 2
                print(f"     Study ID: {ecg.get('study_id', 'N/A')}")
                print(f"       Time: {ecg.get('ecg_time', 'N/A')}")
                if ecg.get('report'):
                    report_preview = ecg['report'][:100] + '...' if len(ecg['report']) > 100 else ecg['report']
                    print(f"       Report: {report_preview}")
        
        # Discharge Note
        if record.get('has_discharge_note', False):
            note = record.get('discharge_note', {})
            print(f"\n📝 Discharge Note:")
            if note.get('text'):
                text_preview = note['text'][:200] + '...' if len(note['text']) > 200 else note['text']
                print(f"   {text_preview}")
        
        # Radiology Notes
        if record.get('has_radiology_note', False):
            rad_notes = record.get('radiology_notes', [])
            print(f"\n📄 Radiology Notes: {len(rad_notes)}")
            for rad in rad_notes[:2]:  # Show first 2
                print(f"     Note ID: {rad.get('note_id', 'N/A')}")
                if rad.get('matched_cxr_study_id'):
                    print(f"       Matched to CXR Study: {rad['matched_cxr_study_id']}")
                if rad.get('text_preview'):
                    print(f"       Preview: {rad['text_preview']}")
    
    print("\n" + "=" * 80)
    print("END OF SAMPLE PATIENT RECORD")
    print("=" * 80)


In [34]:
# Ensure icd-mappings is installed and import Mapper
try:
    from icdmappings import Mapper
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "icd-mappings"])  # quiet install
    from icdmappings import Mapper



In [35]:
# ICD → CCS/CCSR mapping helpers (no embeddings)
import re

def build_ccs_maps(ccs_txt_path: str):
    """
    Parse AppendixASingleDX.txt to build:
      - ccs_code_to_desc: CCS code -> description
      - icd9_to_ccs: ICD-9 code -> set of CCS codes
    """
    ccs_code_to_desc = {}
    icd9_to_ccs = {}
    current_ccs = None
    with open(ccs_txt_path, 'r') as f:
        for raw in f:
            line = raw.rstrip('\n')
            if not line.strip():
                continue
            m = re.match(r'^(\d+)\s+(.+)$', line)
            if m:
                # CCS header line
                current_ccs = m.group(1).strip()
                desc = m.group(2).strip()
                ccs_code_to_desc[current_ccs] = desc
                continue
            # ICD-9 code lines are typically indented list of codes separated by spaces
            if current_ccs is not None:
                parts = line.split()
                for code in parts:
                    code_clean = code.strip()
                    if not code_clean:
                        continue
                    s = icd9_to_ccs.setdefault(code_clean, set())
                    s.add(current_ccs)
    return ccs_code_to_desc, icd9_to_ccs


def build_ccsr_maps(ccsr_csv_path: str):
    """
    Parse DXCCSR_v2025-1.csv to build:
      - ccsr_code_to_desc: CCSR code -> description (first non-empty seen)
      - icd10_to_ccsr: ICD-10-CM code -> set of CCSR codes (union across Default IP/OP and CATEGORY 1..6)
    Uses a robust manual CSV parser to handle quoted fields (aligned with create_embedding_map_icd.py).
    """
    code_desc_pairs = set()
    icd10_to_ccsr = {}

    category_column_indices = [
        (2, 3),  # Default IP
        (4, 5),  # Default OP
        (6, 7),  # CCSR CATEGORY 1
        (8, 9),
        (10, 11),
        (12, 13),
        (14, 15),
        (16, 17),
    ]

    with open(ccsr_csv_path, 'r', encoding='utf-8') as f:
        header = f.readline()
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            fields = []
            current = ''
            in_quote = False
            quote_char = ''
            for ch in line:
                if not in_quote:
                    if ch in ('"', "'"):
                        in_quote = True
                        quote_char = ch
                        current = ''
                    elif ch == ',':
                        fields.append(current.strip())
                        current = ''
                    else:
                        current += ch
                else:
                    if ch == quote_char:
                        in_quote = False
                    elif ch == ',' and not in_quote:
                        fields.append(current.strip())
                        current = ''
                    else:
                        current += ch
            if current:
                fields.append(current.strip())
            if len(fields) < 18:
                continue

            icd10 = fields[0].strip().strip('"').strip("'")
            if not icd10:
                continue
            for code_idx, desc_idx in category_column_indices:
                if code_idx < len(fields) and desc_idx < len(fields):
                    code = fields[code_idx].strip()
                    desc = fields[desc_idx].strip()
                    if code and desc and code != ' ':
                        code_desc_pairs.add((code, desc))
                        icd10_to_ccsr.setdefault(icd10, set()).add(code)

    ccsr_code_to_desc = {}
    for code, desc in code_desc_pairs:
        if code not in ccsr_code_to_desc:
            ccsr_code_to_desc[code] = desc
    return ccsr_code_to_desc, icd10_to_ccsr


def enrich_records_with_icd_categories(records: List[Dict], 
                                       ccs_desc: Dict[str, str], icd9_to_ccs: Dict[str, set] = None,
                                       ccsr_desc: Dict[str, str] = None, icd10_to_ccsr: Dict[str, set] = None,
                                       mapper: Mapper = None) -> List[Dict]:
    """
    For each real admission record with 'diagnoses', add:
      - 'ccs_codes', 'ccs_descs' from ICD-9 codes
      - 'ccsr_codes', 'ccsr_descs' from ICD-10 codes
    Virtual visits (no EHR) remain unchanged.
    """
    enriched = []
    for rec in records:
        r = dict(rec)
        if r.get('has_ehr', False):
            diag_list = r.get('diagnoses', []) or []
            ccs_codes_set = []
            ccsr_codes_set = []
            seen_ccs = set()
            seen_ccsr = set()
            for d in diag_list:
                code = str(d.get('icd_code', '')).strip()
                version = int(d.get('icd_version', 0)) if d.get('icd_version') is not None else 0
                if not code:
                    continue
                if version == 9:
                    mapped = None
                    if mapper is not None:
                        try:
                            mapped = mapper.map(code, source='icd9', target='ccs')
                        except Exception:
                            mapped = None
                    if mapped is None:
                        mapped = list(icd9_to_ccs.get(code, [])) if icd9_to_ccs else []
                    for ccs in (mapped if isinstance(mapped, list) else [mapped]):
                        if not ccs:
                            continue
                        if ccs not in seen_ccs:
                            seen_ccs.add(ccs)
                            ccs_codes_set.append(ccs)
                elif version == 10:
                    mapped = None
                    if mapper is not None:
                        try:
                            mapped = mapper.map(code, source='icd10', target='ccsr')
                        except Exception:
                            mapped = None
                    if mapped is None:
                        mapped = list(icd10_to_ccsr.get(code, [])) if icd10_to_ccsr else []
                    for ccsr in (mapped if isinstance(mapped, list) else [mapped]):
                        if not ccsr:
                            continue
                        if ccsr not in seen_ccsr:
                            seen_ccsr.add(ccsr)
                            ccsr_codes_set.append(ccsr)
            r['ccs_codes'] = ccs_codes_set
            r['ccs_descs'] = [ccs_desc.get(c, None) for c in ccs_codes_set]
            r['ccsr_codes'] = ccsr_codes_set
            r['ccsr_descs'] = [ccsr_desc.get(c, None) for c in ccsr_codes_set]
        enriched.append(r)
    return enriched



## Save and Load Functions

Functions to save and load matching results to avoid re-running the expensive matching process.


In [36]:
def save_matching_results(matched_records: List[Dict], stats: Dict, cache_path: Path = MATCHING_CACHE_PATH, 
                         stats_path: Path = MATCHING_STATS_PATH):
    """
    Save matching results to disk.
    
    Args:
        matched_records: List of matched admission records
        stats: Statistics dictionary
        cache_path: Path to save matching results (pickle)
        stats_path: Path to save statistics (JSON)
    """
    print("=" * 80)
    print("Saving Matching Results")
    print("=" * 80)
    
    # Save matching results (pickle for efficiency)
    print(f"\nSaving matched records to: {cache_path}")
    with open(cache_path, 'wb') as f:
        pickle.dump(matched_records, f)
    print(f"  ✅ Saved {len(matched_records):,} matched records")
    
    # Save statistics (JSON for readability)
    print(f"\nSaving statistics to: {stats_path}")
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2, default=str)
    print(f"  ✅ Saved statistics")
    
    print("\n" + "=" * 80)


def load_matching_results(cache_path: Path = MATCHING_CACHE_PATH, 
                         stats_path: Path = MATCHING_STATS_PATH) -> Tuple[List[Dict], Dict]:
    """
    Load matching results from disk.
    
    Args:
        cache_path: Path to load matching results from (pickle)
        stats_path: Path to load statistics from (JSON)
        
    Returns:
        Tuple of (matched_records, stats)
    """
    print("=" * 80)
    print("Loading Matching Results")
    print("=" * 80)
    
    if not cache_path.exists():
        raise FileNotFoundError(f"Matching results not found at {cache_path}. Please run matching first.")
    
    print(f"\nLoading matched records from: {cache_path}")
    with open(cache_path, 'rb') as f:
        matched_records = pickle.load(f)
    print(f"  ✅ Loaded {len(matched_records):,} matched records")
    
    stats = None
    if stats_path.exists():
        print(f"\nLoading statistics from: {stats_path}")
        with open(stats_path, 'r') as f:
            stats = json.load(f)
        print(f"  ✅ Loaded statistics")
    
    print("\n" + "=" * 80)
    return matched_records, stats


def matching_results_exist(cache_path: Path = MATCHING_CACHE_PATH) -> bool:
    """
    Check if matching results exist.
    
    Args:
        cache_path: Path to check for matching results
        
    Returns:
        True if results exist, False otherwise
    """
    return cache_path.exists()


## Main Pipeline

Run the complete preprocessing pipeline: load data, match modalities, compute statistics, and save results.


In [37]:
# v0/v1 aware main pipeline

# v0 presence check
V0_EXISTS = V0_MATCHING_PATH.exists() and V0_STATS_PATH.exists() and V0_UNASSIGNED_PATH.exists()

if V0_EXISTS:
    print("✅ Found v0 snapshot. Loading v0...")
    with open(V0_MATCHING_PATH, 'rb') as f:
        matched_records = pickle.load(f)
    with open(V0_STATS_PATH, 'r') as f:
        stats = json.load(f)
    with open(V0_UNASSIGNED_PATH, 'rb') as f:
        unassigned_by_patient = pickle.load(f)
    matched_records = sort_matched_records(matched_records)
    print_statistics(stats, matched_records)
else:
    print("🔄 v0 not found. Running matching to produce v0...")

    # Step 1: Load all data
    print("\n" + "=" * 80)
    print("STEP 1: Loading Data")
    print("=" * 80)

    admissions, patients, diagnoses, _, _, _ = load_ehr_data(lazy=True)
    cxr_metadata, _, _, _ = load_cxr_data(lazy=True)
    cxr_reports_map = load_cxr_reports()
    _, _, ecg_report, _ = load_ecg_data(lazy=True)
    discharge_notes, radiology_notes, _, _ = load_notes_data(lazy=True)

    # Step 2: (Optional) Estimate ECG per-patient offsets
    ENABLE_ECG_OFFSET_ESTIMATION = True
    ecg_offsets_by_patient = None
    if ENABLE_ECG_OFFSET_ESTIMATION:
        print("\n" + "=" * 80)
        print("STEP 2: Estimating ECG Time Offsets")
        print("=" * 80)
        ecg_offsets_by_patient = estimate_ecg_time_offsets(
            admissions=admissions,
            ecg_report=ecg_report,
            grid_min_hours=-168,
            grid_max_hours=168,
            grid_step_hours=6,
            min_lift_events=5,
            min_relative_lift=0.25,
            max_abs_hours_accept=168,
            admission_time_slack_hours=6,
        )
        print(f"  Patients with accepted ECG offsets: {len(ecg_offsets_by_patient):,}")
        print(f"\nSaving ECG offset report to: {ECG_OFFSET_REPORT_PATH}")
        with open(ECG_OFFSET_REPORT_PATH, 'w') as f:
            json.dump(ecg_offsets_by_patient, f, indent=2)
        print("  ✅ Saved ECG offset report")

    # Step 3: Matching
    print("\n" + "=" * 80)
    print("STEP 3: Matching Modalities")
    print("=" * 80)

    matched_records = match_modalities_to_admissions(
        admissions=admissions,
        cxr_metadata=cxr_metadata,
        ecg_report=ecg_report,
        discharge_notes=discharge_notes,
        radiology_notes=radiology_notes,
        diagnoses=diagnoses,
        cxr_reports_map=cxr_reports_map,
        time_window_hours=24,
        admission_time_slack_hours=6,
        ecg_offsets_by_patient=ecg_offsets_by_patient,
    )

    matched_records = sort_matched_records(matched_records)

    # Step 3a: Stats
    print("\n" + "=" * 80)
    print("STEP 3a: Computing Statistics")
    print("=" * 80)

    stats = compute_statistics(matched_records)
    print_statistics(stats, matched_records)

    # Step 3b: Unassigned
    print("\n" + "=" * 80)
    print("STEP 3b: Collecting Unassigned Events (no hadm_id)")
    print("=" * 80)
    unassigned_by_patient = collect_unassigned_events(
        admissions=admissions,
        cxr_metadata=cxr_metadata,
        ecg_report=ecg_report,
        radiology_notes=radiology_notes,
        matched_records=matched_records,
        cxr_reports_map=cxr_reports_map,
        admission_time_slack_hours=6,
    )
    num_patients_unassigned = len(unassigned_by_patient)
    total_unassigned_cxr = sum(len(v['cxr_studies']) for v in unassigned_by_patient.values())
    total_unassigned_ecg = sum(len(v['ecg_records']) for v in unassigned_by_patient.values())
    total_unassigned_rad = sum(len(v['radiology_notes']) for v in unassigned_by_patient.values())
    print(f"  Patients with unassigned events: {num_patients_unassigned:,}")
    print(f"  Unassigned CXR studies: {total_unassigned_cxr:,}")
    print(f"  Unassigned ECG records: {total_unassigned_ecg:,}")
    print(f"  Unassigned radiology notes: {total_unassigned_rad:,}")

    # Step 4: Save v0 snapshot
    print("\n" + "=" * 80)
    print("STEP 4: Saving v0 Snapshot")
    print("=" * 80)
    save_matching_results(matched_records, stats)
    with open(PATIENT_UNASSIGNED_PATH, 'wb') as f:
        pickle.dump(unassigned_by_patient, f)
    import shutil
    shutil.copyfile(MATCHING_CACHE_PATH, V0_MATCHING_PATH)
    shutil.copyfile(MATCHING_STATS_PATH, V0_STATS_PATH)
    shutil.copyfile(PATIENT_UNASSIGNED_PATH, V0_UNASSIGNED_PATH)
    if Path(ECG_OFFSET_REPORT_PATH).exists():
        shutil.copyfile(ECG_OFFSET_REPORT_PATH, V0_ECG_OFFSET_PATH)
    print("  ✅ v0 saved")

# Build v1 from v0
print("\n" + "=" * 80)
print("STEP 5: Building v1 (virtual visits)")
print("=" * 80)

with open(V0_MATCHING_PATH, 'rb') as f:
    v0_matched = pickle.load(f)
with open(V0_UNASSIGNED_PATH, 'rb') as f:
    v0_unassigned = pickle.load(f)

v1_records = build_virtual_visits_from_unassigned(
    matched_records=v0_matched,
    unassigned_by_patient=v0_unassigned,
    max_virtual_per_gap=4,
    max_virtual_pre=3,
    max_virtual_post=3,
    min_events_per_virtual=2,
)

v1_stats = compute_statistics(v1_records)
print_statistics(v1_stats, v1_records)

with open(V1_MATCHING_PATH, 'wb') as f:
    pickle.dump(v1_records, f)
with open(V1_STATS_PATH, 'w') as f:
    json.dump(v1_stats, f, indent=2, default=str)
print("  ✅ v1 saved")



✅ Found v0 snapshot. Loading v0...

MATCHING STATISTICS

📊 Total Admissions: 431,231
📊 Unique Patients: 180,733

--------------------------------------------------------------------------------
Modality Presence:
--------------------------------------------------------------------------------
  Ehr                 :  430,852 (99.91%)
  Cxr                 :   74,654 (17.31%)
  Ecg                 :  207,422 (48.10%)
  Discharge Note      :  331,793 (76.94%)
  Radiology Note      :  229,894 (53.31%)

--------------------------------------------------------------------------------
Modality Combinations (Top 10):
--------------------------------------------------------------------------------
  Discharge+EHR                 :   92,119 (21.36%)
  Discharge+ECG+EHR+Radiology   :   85,508 (19.83%)
  EHR                           :   56,007 (12.99%)
  Discharge+EHR+Radiology       :   51,223 (11.88%)
  CXR+Discharge+ECG+EHR+Radiology:   48,125 (11.16%)
  Discharge+ECG+EHR             :   38,4

In [38]:
# v0 vs v1 comparison and v1 structure summary
import json, pickle
import numpy as np
from collections import defaultdict, Counter

print("\n" + "=" * 80)
print("V0 vs V1: Patient-level Admission Distribution (with percentiles)")
print("=" * 80)

# Load v0/v1 stats
with open(V0_STATS_PATH, 'r') as f:
    v0_stats = json.load(f)
with open(V1_STATS_PATH, 'r') as f:
    v1_stats = json.load(f)

# Extract distributions
v0_adm_per_patient = v0_stats['sequence_lengths']['admissions_per_patient']
v1_adm_per_patient = v1_stats['sequence_lengths']['admissions_per_patient']

# Percentiles helper
percentiles = [25, 50, 75, 90, 95, 99]

def pct_summary(values):
    arr = np.array(values, dtype=float)
    return {f"p{p}": float(np.percentile(arr, p)) for p in percentiles}

v0_summary = {
    'patients': int(np.sum(list(v0_stats['sequence_lengths']['by_patient'].values()))),
    'total_admissions': v0_stats['total_admissions'],
    'mean': float(np.mean(v0_adm_per_patient)),
    'median': float(np.median(v0_adm_per_patient)),
    'percentiles': pct_summary(v0_adm_per_patient),
    'pct_one_adm': 100.0 * (v0_stats['sequence_lengths']['by_patient'].get('1', 0) / max(1, np.sum(list(v0_stats['sequence_lengths']['by_patient'].values()))))
}

v1_summary = {
    'patients': int(np.sum(list(v1_stats['sequence_lengths']['by_patient'].values()))),
    'total_admissions': v1_stats['total_admissions'],
    'mean': float(np.mean(v1_adm_per_patient)),
    'median': float(np.median(v1_adm_per_patient)),
    'percentiles': pct_summary(v1_adm_per_patient),
    'pct_one_adm': 100.0 * (v1_stats['sequence_lengths']['by_patient'].get('1', 0) / max(1, np.sum(list(v1_stats['sequence_lengths']['by_patient'].values()))))
}

print("v0:", v0_summary)
print("v1:", v1_summary)

print("\n" + "=" * 80)
print("V1 Structure: Virtual vs Real & Virtual Groups")
print("=" * 80)

# Load v1 records
with open(V1_MATCHING_PATH, 'rb') as f:
    v1_records = pickle.load(f)

num_total = len(v1_records)
num_real = sum(1 for r in v1_records if not r.get('is_virtual'))
num_virtual = num_total - num_real

vg_counts = Counter(r.get('virtual_group', 'real') for r in v1_records)

# Per-patient virtual counts
per_patient = defaultdict(lambda: {'total': 0, 'virtual': 0})
for r in v1_records:
    pid = r['subject_id']
    per_patient[pid]['total'] += 1
    if r.get('is_virtual'):
        per_patient[pid]['virtual'] += 1

virt_counts = [v['virtual'] for v in per_patient.values()]
virt_any = [v for v in virt_counts if v > 0]

virt_pctiles_all = pct_summary(virt_counts)
virt_pctiles_any = pct_summary(virt_any) if virt_any else {}

print({
    'total_admissions': num_total,
    'real_admissions': num_real,
    'virtual_admissions': num_virtual,
    'virtual_group_counts': dict(vg_counts),
    'patients_total': len(per_patient),
    'patients_with_virtual': sum(1 for v in per_patient.values() if v['virtual'] > 0),
    'virtual_per_patient_percentiles_all': virt_pctiles_all,
    'virtual_per_patient_percentiles_cond_any': virt_pctiles_any,
})

print("\n" + "=" * 80)
print("Unassigned Events Consumption in V1 Virtual Visits")
print("=" * 80)

# From v0 unassigned
with open(V0_UNASSIGNED_PATH, 'rb') as f:
    v0_unassigned = pickle.load(f)

u_cxr = sum(len(v.get('cxr_studies', [])) for v in v0_unassigned.values())
u_ecg = sum(len(v.get('ecg_records', [])) for v in v0_unassigned.values())
u_rad = sum(len(v.get('radiology_notes', [])) for v in v0_unassigned.values())

# Count events used in v1 virtual visits
vv = [r for r in v1_records if r.get('is_virtual')]
vv_cxr = sum(len(r.get('cxr_studies', [])) for r in vv)
vv_ecg = sum(len(r.get('ecg_records', [])) for r in vv)
vv_rad = sum(len(r.get('radiology_notes', [])) for r in vv)

left_cxr = max(0, u_cxr - vv_cxr)
left_ecg = max(0, u_ecg - vv_ecg)
left_rad = max(0, u_rad - vv_rad)

def pct(a, b):
    return 100.0 * a / b if b else 0.0

print({
    'v0_unassigned': {'cxr_studies': u_cxr, 'ecg_records': u_ecg, 'radiology_notes': u_rad},
    'v1_virtual_used': {'cxr_studies': vv_cxr, 'ecg_records': vv_ecg, 'radiology_notes': vv_rad},
    'leftover_after_v1': {'cxr_studies': left_cxr, 'ecg_records': left_ecg, 'radiology_notes': left_rad},
    'assigned_pct': {'cxr_studies': pct(vv_cxr, u_cxr), 'ecg_records': pct(vv_ecg, u_ecg), 'radiology_notes': pct(vv_rad, u_rad)},
})




V0 vs V1: Patient-level Admission Distribution (with percentiles)
v0: {'patients': 180733, 'total_admissions': 431231, 'mean': 2.3860114090951847, 'median': 1.0, 'percentiles': {'p25': 1.0, 'p50': 1.0, 'p75': 2.0, 'p90': 5.0, 'p95': 7.0, 'p99': 16.0}, 'pct_one_adm': np.float64(55.99309478623162)}
v1: {'patients': 204637, 'total_admissions': 606089, 'mean': 2.9617762183769307, 'median': 2.0, 'percentiles': {'p25': 1.0, 'p50': 2.0, 'p75': 3.0, 'p90': 6.0, 'p95': 10.0, 'p99': 21.0}, 'pct_one_adm': np.float64(49.62005893362393)}

V1 Structure: Virtual vs Real & Virtual Groups
{'total_admissions': 606089, 'real_admissions': 431231, 'virtual_admissions': 174858, 'virtual_group_counts': {'real': 431231, 'post': 31686, 'pre': 70482, 'between': 72690}, 'patients_total': 204637, 'patients_with_virtual': 79835, 'virtual_per_patient_percentiles_all': {'p25': 0.0, 'p50': 0.0, 'p75': 1.0, 'p90': 3.0, 'p95': 4.0, 'p99': 8.0}, 'virtual_per_patient_percentiles_cond_any': {'p25': 1.0, 'p50': 1.0, 'p75'

## Display Sample Patient

Display a sample patient record to verify the matching results.


In [39]:
# Display a sample patient with multiple modalities
display_sample_patient(matched_records, subject_id=None, min_modalities=2, max_admissions=3)


SAMPLE PATIENT RECORD

Patient ID: 10000032
Total Admissions: 4
Displaying: 3 admission(s)

ADMISSION #1: HADM ID 22595853

📅 Admission Period:
   Admit:    2180-05-06 22:23:00
   Discharge: 2180-05-07 17:15:00
   Duration: 0 days 18:52:00
   Type:     URGENT

📋 Modalities: EHR, CXR, Discharge Note, Radiology Note

🏥 EHR Data:
   Diagnoses: 8
     [1] ICD-9: 5723
     [2] ICD-9: 78959
     [3] ICD-9: 5715
     [4] ICD-9: 07070
     [5] ICD-9: 496
     ... and 3 more

📷 CXR Data:
   Studies: 1
   Total Images: 2
     Study ID: 50414267
       Images: 2
       First image: 02aa804e-bde0afdd-112c0b34-7bc16630-4e38...
       Time: 2180-05-06T21:30:14

📝 Discharge Note:
    
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
No Known Allergie...

📄 Radiology Notes: 1
     Note ID: 10000032-RR-14
       Matched to CXR Study: 50414267
       Preview: EXAMINATION:  

In [40]:
# Enrich existing v1 records with CCS/CCSR (safe to re-run)
print("\n" + "=" * 80)
print("Enriching v1 records with CCS/CCSR categories (post-build)")
print("=" * 80)

# Load v1
with open(V1_MATCHING_PATH, 'rb') as f:
    v1_records = pickle.load(f)

# Build maps
ccs_desc_map, icd9_to_ccs_map = build_ccs_maps("AppendixASingleDX.txt")
ccsr_desc_map, icd10_to_ccsr_map = build_ccsr_maps("DXCCSR_v2025-1.csv")

# Enrich and save
v1_records = enrich_records_with_icd_categories(v1_records, ccs_desc_map, icd9_to_ccs_map,
                                                ccsr_desc_map, icd10_to_ccsr_map)

v1_stats = compute_statistics(v1_records)
print_statistics(v1_stats, v1_records)

with open(V1_MATCHING_PATH, 'wb') as f:
    pickle.dump(v1_records, f)
with open(V1_STATS_PATH, 'w') as f:
    json.dump(v1_stats, f, indent=2, default=str)
print("  ✅ v1 enriched and saved")




Enriching v1 records with CCS/CCSR categories (post-build)
Computing Statistics

MATCHING STATISTICS

📊 Total Admissions: 606,089
📊 Unique Patients: 204,637

--------------------------------------------------------------------------------
Modality Presence:
--------------------------------------------------------------------------------
  Ehr                 :  430,852 (71.09%)
  Cxr                 :  108,624 (17.92%)
  Ecg                 :  332,044 (54.78%)
  Discharge Note      :  331,793 (54.74%)
  Radiology Note      :  364,912 (60.21%)

--------------------------------------------------------------------------------
Modality Combinations (Top 10):
--------------------------------------------------------------------------------
  Discharge+EHR                 :   92,119 (15.20%)
  Discharge+ECG+EHR+Radiology   :   85,508 (14.11%)
  ECG+Radiology                 :   69,437 (11.46%)
  EHR                           :   56,007 ( 9.24%)
  Discharge+EHR+Radiology       :   51,223 ( 8.

In [41]:
# Enrich existing v0 records with CCS/CCSR (optional post-build)
print("\n" + "=" * 80)
print("Enriching v0 records with CCS/CCSR categories (post-build)")
print("=" * 80)

# Load v0
with open(V0_MATCHING_PATH, 'rb') as f:
    v0_records = pickle.load(f)

# Build description maps and use Mapper for code→category mapping
mapper = Mapper()
ccs_desc_map, _ = build_ccs_maps("AppendixASingleDX.txt")
ccsr_desc_map, _ = build_ccsr_maps("DXCCSR_v2025-1.csv")

# Enrich and save (mapper provides ICD→CCS/CCSR mapping)
v0_records = enrich_records_with_icd_categories(v0_records, ccs_desc_map, None,
                                                ccsr_desc_map, None, mapper=mapper)

v0_stats = compute_statistics(v0_records)
print_statistics(v0_stats, v0_records)

with open(V0_MATCHING_PATH, 'wb') as f:
    pickle.dump(v0_records, f)
with open(V0_STATS_PATH, 'w') as f:
    json.dump(v0_stats, f, indent=2, default=str)
print("  ✅ v0 enriched and saved")




Enriching v0 records with CCS/CCSR categories (post-build)
Computing Statistics

MATCHING STATISTICS

📊 Total Admissions: 431,231
📊 Unique Patients: 180,733

--------------------------------------------------------------------------------
Modality Presence:
--------------------------------------------------------------------------------
  Ehr                 :  430,852 (99.91%)
  Cxr                 :   74,654 (17.31%)
  Ecg                 :  207,422 (48.10%)
  Discharge Note      :  331,793 (76.94%)
  Radiology Note      :  229,894 (53.31%)

--------------------------------------------------------------------------------
Modality Combinations (Top 10):
--------------------------------------------------------------------------------
  Discharge+EHR                 :   92,119 (21.36%)
  Discharge+ECG+EHR+Radiology   :   85,508 (19.83%)
  EHR                           :   56,007 (12.99%)
  Discharge+EHR+Radiology       :   51,223 (11.88%)
  CXR+Discharge+ECG+EHR+Radiology:   48,125 (11

## Query Functions

Helper functions to query matched records by flags and criteria.


In [42]:
def filter_records_by_flags(matched_records: List[Dict], 
                            has_ehr: Optional[bool] = None,
                            has_cxr: Optional[bool] = None,
                            has_ecg: Optional[bool] = None,
                            has_discharge_note: Optional[bool] = None,
                            has_radiology_note: Optional[bool] = None,
                            min_admissions_per_patient: int = 1,
                            max_admissions_per_patient: int = None) -> List[Dict]:
    """
    Filter matched records by modality flags and sequence length.
    
    Args:
        matched_records: List of matched admission records
        has_ehr: Filter by EHR presence (None = no filter)
        has_cxr: Filter by CXR presence (None = no filter)
        has_ecg: Filter by ECG presence (None = no filter)
        has_discharge_note: Filter by discharge note presence (None = no filter)
        has_radiology_note: Filter by radiology note presence (None = no filter)
        min_admissions_per_patient: Minimum number of admissions per patient
        max_admissions_per_patient: Maximum number of admissions per patient
        
    Returns:
        Filtered list of matched records
    """
    filtered = matched_records.copy()
    
    # Filter by modality flags
    if has_ehr is not None:
        filtered = [r for r in filtered if r.get('has_ehr', False) == has_ehr]
    if has_cxr is not None:
        filtered = [r for r in filtered if r.get('has_cxr', False) == has_cxr]
    if has_ecg is not None:
        filtered = [r for r in filtered if r.get('has_ecg', False) == has_ecg]
    if has_discharge_note is not None:
        filtered = [r for r in filtered if r.get('has_discharge_note', False) == has_discharge_note]
    if has_radiology_note is not None:
        filtered = [r for r in filtered if r.get('has_radiology_note', False) == has_radiology_note]
    
    # Filter by sequence length (admissions per patient)
    if min_admissions_per_patient > 1 or max_admissions_per_patient is not None:
        patient_admissions = defaultdict(list)
        for record in filtered:
            patient_admissions[record['subject_id']].append(record)
        
        valid_patients = set()
        for patient_id, admissions in patient_admissions.items():
            num_adm = len(admissions)
            if num_adm >= min_admissions_per_patient:
                if max_admissions_per_patient is None or num_adm <= max_admissions_per_patient:
                    valid_patients.add(patient_id)
        
        filtered = [r for r in filtered if r['subject_id'] in valid_patients]
    
    return filtered


# Example: Find patients with all three modalities (EHR, CXR, ECG)
print("Example Query: Patients with all three modalities (EHR, CXR, ECG)")
print("=" * 80)

all_three_modalities = filter_records_by_flags(
    matched_records,
    has_ehr=True,
    has_cxr=True,
    has_ecg=True
)

print(f"\nFound {len(all_three_modalities):,} admissions with all three modalities")
if len(all_three_modalities) > 0:
    unique_patients = len(set(r['subject_id'] for r in all_three_modalities))
    print(f"From {unique_patients:,} unique patients")
    
    # Show one example
    if len(all_three_modalities) > 0:
        print("\nExample admission:")
        sample = all_three_modalities[0]
        print(f"  Patient: {sample['subject_id']}, HADM: {sample['hadm_id']}")
        print(f"  Modalities: EHR={sample.get('has_ehr')}, CXR={sample.get('has_cxr')}, ECG={sample.get('has_ecg')}")


Example Query: Patients with all three modalities (EHR, CXR, ECG)

Found 57,352 admissions with all three modalities
From 34,375 unique patients

Example admission:
  Patient: 10000032, HADM: 29079034
  Modalities: EHR=True, CXR=True, ECG=True


In [43]:
from typing import Iterable

def group_records_by_patient(matched_records: List[Dict]) -> Dict[int, List[Dict]]:
    """
    Group admissions by subject_id.
    """
    patients: Dict[int, List[Dict]] = defaultdict(list)
    for record in matched_records:
        patients[record["subject_id"]].append(record)
    # Sort each patient's admissions chronologically
    for pid in patients:
        patients[pid] = sorted(patients[pid], key=lambda r: r["admittime"])
    return patients


def find_patients_with_min_modalities(matched_records: List[Dict], min_modalities: int = 2) -> List[int]:
    """
    Return subject_ids that have at least min_modalities across their admissions.
    """
    patients = group_records_by_patient(matched_records)
    qualified: List[int] = []
    for pid, admissions in patients.items():
        modalities = set()
        for adm in admissions:
            if adm.get("has_ehr", False):
                modalities.add("EHR")
            if adm.get("has_cxr", False):
                modalities.add("CXR")
            if adm.get("has_ecg", False):
                modalities.add("ECG")
            if adm.get("has_discharge_note", False):
                modalities.add("Discharge")
            if adm.get("has_radiology_note", False):
                modalities.add("Radiology")
        if len(modalities) >= min_modalities:
            qualified.append(pid)
    return qualified


def display_query_examples(
    matched_records: List[Dict],
    filters: Dict,
    max_patients: int = 2,
    max_admissions: int = 3,
) -> None:
    """
    Apply filter_records_by_flags and display a few example patients.

    Args:
        matched_records: Full list of admission records
        filters: Keyword arguments to pass to filter_records_by_flags
        max_patients: Number of distinct patients to display
        max_admissions: Max admissions per displayed patient
    """
    print("=" * 80)
    print("QUERY EXAMPLES")
    print("=" * 80)

    filtered = filter_records_by_flags(matched_records, **filters)
    if len(filtered) == 0:
        print("No admissions matched the given filters.")
        return

    # Choose distinct patients from filtered admissions
    seen: set = set()
    chosen_subjects: List[int] = []
    for rec in filtered:
        pid = rec["subject_id"]
        if pid not in seen:
            seen.add(pid)
            chosen_subjects.append(pid)
            if len(chosen_subjects) >= max_patients:
                break

    print(f"Matched admissions: {len(filtered):,}")
    print(f"Example patients to display: {len(chosen_subjects)}")

    for i, pid in enumerate(chosen_subjects, 1):
        print(f"\n--- Example Patient {i} (subject_id={pid}) ---")
        display_sample_patient(matched_records, subject_id=pid, max_admissions=max_admissions)

